In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.seasonal import STL

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().resolve().parent))
print(sys.executable)

from src.metrics import *
print("ok")

/Users/yuanxinyi/Desktop/panel-forcast-kit/.venv/bin/python
ok


In [5]:
project_root = Path.cwd().resolve().parent if "panel-forcast-kit" in str(Path.cwd()) else Path.cwd()

processed_dir = project_root / "data" / "processed"

X_train = pd.read_csv(processed_dir / "X_train.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv")
X_test = pd.read_csv(processed_dir / "X_test.csv")
y_test = pd.read_csv(processed_dir / "y_test.csv")

# 统一为 Series / DataFrame 的形式
X_train = X_train.copy()
y_train = y_train.iloc[:, 0].copy()
X_test = X_test.copy()
y_test = y_test.iloc[:, 0].copy()

print("Loaded:")
print(f"X_train shape: {X_train.shape}, \n{X_train.head(1)}")
print(f"\ny_train shape: {y_train.shape}, \n{y_train.head(1)}")
print(f"\nX_test shape: {X_test.shape}, \n{X_test.head(1)}")
print(f"\ny_test shape: {y_test.shape}, \n{y_test.head(1)}")

Loaded:
X_train shape: (853000, 11), 
   store  item  day  month  is_weekend  is_holiday  lag_1  lag_7  lag_14  \
0      1     1   29      1           0           0   11.0    7.0     5.0   

   lag_28  rolling_mean_7  
0    13.0       10.428571  

y_train shape: (853000,), 
0    6
Name: sales, dtype: int64

X_test shape: (46000, 11), 
   store  item  day  month  is_weekend  is_holiday  lag_1  lag_7  lag_14  \
0      1     1    1     10           1           0   30.0   24.0    30.0   

   lag_28  rolling_mean_7  
0    23.0       23.428571  

y_test shape: (46000,), 
0    21
Name: sales, dtype: int64


# Baseline

In [6]:
# Baseline: direct prediction and evaluation on the already split test set
baseline_cols = {
    "Same period last week (lag7)": "lag_7",
    "7-day moving average (MA7)": "rolling_mean_7"
}

print(f"\n=== Baseline performance on the test set (sample count: {len(X_test)}) ===")
baseline_results = {}
for name, col in baseline_cols.items():
    baseline_results[name] = evaluate(y_test, X_test[col])
    mae, rmse, wape = baseline_results[name]
    print(f"{name:20s}  MAE: {mae:6.3f}   RMSE: {rmse:6.3f}   WAPE: {wape:6.2%}")


=== Baseline performance on the test set (sample count: 46000) ===
Same period last week (lag7)  MAE:  9.076   RMSE: 12.051   WAPE: 16.60%
7-day moving average (MA7)  MAE:  8.712   RMSE: 11.463   WAPE: 15.93%


# Model 1